In [ ]:
import torch
from torch import nn
import torchvision
from torch.utils import data
from tqdm import tqdm
import numpy as np
from datetime import datetime
import json
import wandb
import os
import math
from tqdm import tqdm

from models import DINO_CLIP, TwoCropsTransform, GaussianBlur
from datasets import ImageNet, Places365, ArtPlaces, iNaturalist, Transforms
from evaluation_utils import evaluate_features
import models.vicreg.augmentations as aug
from models.vicreg.main_vicreg import VICReg, LARS

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(torch.cuda.get_device_name())

In [ ]:
LOG_RUN = True

CLIP_MODEL_NAME = "ViT-B/32"
DINO_MODEL_NAME = "dinov2_vitb14"
CONVNEXT_MODEL_NAME = "convnextv2_nano.fcmae_ft_in22k_in1k"
FUSION_HEAD = "LinearSmall"
LAMBDA_ALPHA_REG = 0
USE_WEIGHTED_CONCAT = False
USE_DINO_CLS_AND_PATCH_TOKENS = True
USE_PROJ = False
PROJ_DIM = 512

DATASET = "iNaturalist"
TARGET_TYPE = None
match DATASET:
    case "iNaturalist":
        TARGET_TYPE = "family"

BATCH_SIZE = 64
EPOCHS = 5
# CRITERION = "cross_entropy"
# OPTIMIZER = "adam"
# SCHEDULER = "cosine"
# SCHEDULER_MIN_LR = None
# # match SCHEDULER:
# #     case "step":
# #         SCHEDULE = [15, 30]
# #     case "cosine":
# #         SCHEDULE = None
# match SCHEDULER:
#     case "cosine":
#         SCHEDULER_MIN_LR = 0.00001

# MOMENTUM = 0.9
WEIGHT_DECAY = 1e-6
LR = 0.2
SIM_COEFF = 25.0
STD_COEFF = 25.0
COV_COEFF = 1.0

DIM = 1024 # 512 1024

EVALUATION_DISTANCE = "l2"
EVALUATION_K = [1, 5]

In [ ]:
config = {
    "clip_model_name": CLIP_MODEL_NAME,
    "dino_model_name": DINO_MODEL_NAME,
    "convnext_model_name": CONVNEXT_MODEL_NAME,
    "fusion_head": FUSION_HEAD,
    "lambda_alpha_reg": LAMBDA_ALPHA_REG,
    "use_weighted_concat": USE_WEIGHTED_CONCAT,
    "use_dino_cls_and_patch_tokens": USE_DINO_CLS_AND_PATCH_TOKENS,
    "use_proj": USE_PROJ,
    "proj_dim": PROJ_DIM,
    "dataset": DATASET,
    "target_type": TARGET_TYPE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    # "criterion": CRITERION,
    # "optimizer": OPTIMIZER,
    # "optimizer_momentum": MOMENTUM,
    "optimizer_weight_decay": WEIGHT_DECAY,
    # "scheduler": SCHEDULER,
    # "scheduler_min_lr": SCHEDULER_MIN_LR,
    # "schedule": SCHEDULE,
    "dim": DIM,
}

In [ ]:
if LOG_RUN:
    wandb.login()

In [ ]:
NUM_WORKERS = 8

transform = aug.TrainTransform()
clip_transform = torchvision.transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
dino_transform = torchvision.transforms.Normalize(mean=(0.485, 0.456, 0.406),std=(0.229, 0.224, 0.225))
convnext_transform = torchvision.transforms.Normalize(mean=(0.485, 0.456, 0.406),std=(0.229, 0.224, 0.225))

match DATASET:
    case "ImageNet":
        dataset = ImageNet(root=r"C:\Users\mariu\Documents\Development\Datasets\ImageNet", transform=transform)
        dataset_val = ImageNet(root=r"C:\Users\mariu\Documents\Development\Datasets\ImageNet", transform=transform, split="val")
    case "Places365":
        dataset = Places365(root=r"C:\Users\mariu\Documents\Development\Datasets\Places365", transform=transform)
        dataset_val = Places365(root=r"C:\Users\mariu\Documents\Development\Datasets\Places365", transform=transform, split="val")
    case "ArtPlaces":
        dataset = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280", transform=transform)
        dataset_val = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280", transform=transform, split="val")
    case "iNaturalist":
        dataset = iNaturalist(root=r"D:\Dokumente\Development\Datasets\iNaturalist", transform=transform, target_type=TARGET_TYPE, split="train_mini")
        dataset_val = iNaturalist(root=r"D:\Dokumente\Development\Datasets\iNaturalist", transform=transform, target_type=TARGET_TYPE, split="val")
        dataset_val = dataset_val.getSubset(32000, classes=list(dataset_val.class_to_idx.keys()))

data_loader = data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
data_loader_val = data.DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

In [ ]:
model = VICReg(
    BATCH_SIZE,
    SIM_COEFF,
    STD_COEFF,
    COV_COEFF,
    clip_model_name=CLIP_MODEL_NAME,
    dino_model_name=DINO_MODEL_NAME,
    convnext_model_name = CONVNEXT_MODEL_NAME,
    use_dino_cls_and_patch_tokens = USE_DINO_CLS_AND_PATCH_TOKENS,
    clip_transform = None,
    dino_transform = None,
    convnext_transform = None,
    fusion_head=FUSION_HEAD,
    dim = DIM,
    use_weighted_concat = USE_WEIGHTED_CONCAT,
    use_proj = USE_PROJ,
    proj_dim = PROJ_DIM
)
model = model.train()
model = model.to(device)

In [ ]:
if LOG_RUN:
    config["fusion_head_architecture"] = str(model.fusion_head)
    
    wandb.init(
        # set the wandb project where this run will be logged
        project="vicreg",
        dir=r"C:\Users\mariu\Desktop",

        # track hyperparameters and run metadata
        config=config
    )

    wandb.watch(model.fusion_head, log="all", log_freq=100)

In [ ]:
dest = os.path.join(r"D:\Dokumente\Studium\Masterprojekt\Gewichte", "vicreg", DINO_MODEL_NAME.lower() + "_" + CLIP_MODEL_NAME.lower().replace("/", "") + "_" + DATASET.lower() + "_" + datetime.today().strftime('%Y%m%d_%H%M%S'))
os.makedirs(dest)

with open(os.path.join(dest, "constants.json"), "w") as file:
    json.dump(config, file)

def save_model(epoch=0, batch=0, batch_global=0):
    torch.save(model.state_dict(), os.path.join(dest, "state_dict_epoch_" + str(epoch) + "_batch_" + str(batch) + "_batch_global_" + str(batch_global) + ".pt"))

In [ ]:
def exclude_bias_and_norm(p):
    return p.ndim == 1

optimizer = LARS(
    model.parameters(),
    lr=0,
    weight_decay=WEIGHT_DECAY,
    weight_decay_filter=exclude_bias_and_norm,
    lars_adaptation_filter=exclude_bias_and_norm,
)

In [ ]:
batch_max = len(data_loader) * EPOCHS
batch_global = 0
EVAL_AFTER = 2048 # 1024 # Number of batches after which to evaluate on the validation set (= number of batches in an virtual epoch)
SAVE_AFTER = 10240 # 20480

def adjust_learning_rate(epochs, base_lr, batch_size, optimizer, loader, step):
    max_steps = epochs * len(loader)
    warmup_steps = 10 * len(loader)
    base_lr = base_lr * batch_size / 256
    if step < warmup_steps:
        lr = base_lr * step / warmup_steps
    else:
        step -= warmup_steps
        max_steps -= warmup_steps
        q = 0.5 * (1 + math.cos(math.pi * step / max_steps))
        end_lr = base_lr * 0.001
        lr = base_lr * q + end_lr * (1 - q)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr
    return lr

scaler = torch.amp.GradScaler("cuda")

for epoch in range(EPOCHS):
    losses = []
    val_losses = []

    tqdm_data_loader = tqdm(data_loader, unit="batch")
    tqdm_data_loader.set_description(f"Epoch {epoch+1}/{EPOCHS}")
    for i, ((x, y), _) in enumerate(tqdm_data_loader):
        x = x.to(device)
        y = y.to(device)

        lr = adjust_learning_rate(EPOCHS, LR, BATCH_SIZE, optimizer, data_loader, i)

        # optimizer.zero_grad()
        # with torch.amp.autocast("cuda"):
        #     loss = model.forward(x, y)
        # scaler.scale(loss).backward()
        # scaler.step(optimizer)
        # scaler.update()

        loss = model.forward(x, y)

        if (i+1) % 10 == 0 and LOG_RUN:
            wandb.log({
                "loss": loss.item(),
                "learning_rate": lr, 
                "epoch": epoch + 1,
                "batch": i + 1,
                "batch_global": batch_global,
            })

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        postfix = {
            "Loss": loss.item(),
        }
        tqdm_data_loader.set_postfix(postfix)

        # Val
        if (batch_global + 1) % EVAL_AFTER == 0 or (batch_global + 1) == batch_max:
            result = {}

            model.eval()
            with torch.no_grad():
                features = []
                labels = []

                tqdm_data_loader = tqdm(data_loader, unit="batch")
                tqdm_data_loader.set_description(f"Epoch {epoch+1}/{EPOCHS}")
                for i, ((x, y), labels_batch) in enumerate(tqdm_data_loader):
                    x = x.to(device)
                    y = y.to(device)

                    lr = adjust_learning_rate(EPOCHS, LR, BATCH_SIZE, optimizer, data_loader, i)

                    loss = model.forward(x, y)
                    features_batch = model.forward(x)

                    val_losses.append(loss.item())
                    features.extend(features_batch.cpu().tolist())
                    labels.extend(labels_batch.tolist())
                features = np.array(features).astype("float32")
                labels = np.array(labels)

                result = evaluate_features(features, labels, EVALUATION_DISTANCE, EVALUATION_K)
            model.train()

            if LOG_RUN:
                wandb.log({
                    "loss_avg": np.mean(losses),
                    "val_loss_avg": np.mean(val_losses),
                    **{f"acc@{k}": result[f"Accuracy@{k}"].item() for k in EVALUATION_K},
                    **{f"per_class_mean_acc@{k}": result["per_class"]["mean"][f"Accuracy@{k}"].item() for k in EVALUATION_K},
                    **{f"per_class_mean_precision@{k}": result["per_class"]["mean"][f"Precision@{k}"].item() for k in EVALUATION_K},
                    **{f"per_class_mean_recall@{k}": result["per_class"]["mean"][f"Recall@{k}"].item() for k in EVALUATION_K}
                })

            losses = []
            val_losses = []

        if (batch_global + 1) % SAVE_AFTER == 0 or (batch_global + 1) == batch_max:
            save_model(epoch=epoch + 1, batch=i + 1, batch_global=batch_global + 1)
        
        batch_global += 1

In [ ]:
if LOG_RUN:
    wandb.finish()